# ELT Pipeline: Limpieza y Carga de Datos de E-commerce

Este script realiza la limpieza de datos transaccionales, gestiona valores nulos, corrige tipos de datos y carga 
el dataframe resultante en una base de datos MySQL

In [10]:
import pandas as pd
import numpy as np 

In [14]:
df=pd.read_csv(r"C:\Users\user\Desktop\MYSQL\E-Commerce Data\data.csv", encoding='ISO-8859-1')

In [15]:
print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  


In [13]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None


In [16]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [17]:
df.describe(include='object')

,InvoiceNo,StockCode,Description,InvoiceDate,Country
count,541909,541909,540455,541909,541909
unique,25900,4070,4223,23260,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,10/31/2011 14:41,United Kingdom
freq,1114,2313,2369,1114,495478


## Limpieza de datos 

Eliminamos los registros sin CustomerID y tratamos las cantidades negativas para asegurar la integridad de los KPIs. 

In [35]:
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean[df_clean['UnitPrice'] > 0]
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice'] 
print(df_clean.info())

<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397924 non-null  object        
 1   StockCode    397924 non-null  object        
 2   Description  397924 non-null  object        
 3   Quantity     397924 non-null  int64         
 4   InvoiceDate  397924 non-null  datetime64[ns]
 5   UnitPrice    397924 non-null  float64       
 6   CustomerID   397924 non-null  int64         
 7   Country      397924 non-null  object        
 8   TotalAmount  397924 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 30.4+ MB
None


## Exportacion a MySQL 

Usamos SQLA1chemy para crear una conexion eficiente y cargar los datos en la tabla 'ventas_raw'. 

In [34]:
pip install mysql-connector-python sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [36]:
from sqlalchemy import create_engine

In [ ]:
usuario = "root"
password = "tu_contraseña"
host = 'localhost'
base_de_datos = "ecommerce_db"
engine = create_engine(f"mysql+mysqlconnector://{usuario}:{password}@{host}/{base_de_datos}")
try:
    df_clean.to_sql('ventas_raw', con=engine, if_exists='replace', index=False)
    print("¡Éxito! Los datos están en MySQL.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
import mysql.connector
from sqlalchemy import create_engine


USUARIO = "root"
PASSWORD = "tu_contraseña"  
HOST = 'localhost'          
DB_NOMBRE = "ecommerce_db"

try:
    db_test = mysql.connector.connect(host=HOST, user=USUARIO, password=PASSWORD)
    cursor = db_test.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NOMBRE}")
    db_test.close()
    print("✅ Base de datos verificada/creada.")
except Exception as e:
    print(f"❌ Error al acceder a MySQL: {e}")

engine = create_engine(f"mysql+mysqlconnector://{USUARIO}:{PASSWORD}@{HOST}/{DB_NOMBRE}")

try:
    df_clean.to_sql('ventas_raw', con=engine, if_exists='replace', index=False, chunksize=1000)
    print("✅ ¡Pandas dice que ha enviado los datos a 'ventas_raw'!")
except Exception as e:
    print(f"❌ Error en la carga: {e}")